In [ ]:

# Step 24.1: Install required libraries

%pip -q install google-cloud-bigquery db-dtypes


In [ ]:

# Step 24.2: Authenticate and connect to BigQuery

from google.colab import auth
from google.cloud import bigquery
import pandas as pd

# Authenticate with your Google account
auth.authenticate_user()

# Configure your Google Cloud project
PROJECT_ID = "ga4-ecommerce-analytics-511204"
DATASET_ID = "ga4_analytics"
TABLE_ID = "user_retention_features"

# Create BigQuery client
client = bigquery.Client(
    project=PROJECT_ID,
    location="US"
)

# Full table identifier
table_name = (
    f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
)

# Check if table exists and is accessible
table = client.get_table(table_name)

print("BigQuery connection successful!")
print("Project:", PROJECT_ID)
print("Table:", table_name)
print("Rows:", table.num_rows)
print("Columns:", len(table.schema))


In [ ]:

# Step 24.3: Validate the feature table from Python

sql = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS unique_users,
    SUM(returned_w1) AS retained_users,

    ROUND(
        100 * SAFE_DIVIDE(
            SUM(returned_w1),
            COUNT(*)
        ),
        2
    ) AS retention_rate_pct,

    COUNTIF(
        first_session_id IS NULL
    ) AS missing_session_ids,

    COUNTIF(
        NOT has_matched_events
    ) AS unmatched_users,

    COUNTIF(
        event_count <
        page_views +
        item_views +
        cart_events +
        checkout_events +
        purchase_events
    ) AS invalid_event_subtotals

FROM `{table_name}`
"""

# Execute SQL directly from Python
job = client.query(sql, location="US")

df_validation = job.to_dataframe(
    create_bqstorage_client=False
)

print("=== BigQuery Data Validation ===")
display(df_validation)

# Verify key metrics
assert df_validation.loc[0, "total_rows"] == 181360
assert df_validation.loc[0, "unique_users"] == 181360
assert df_validation.loc[0, "retained_users"] == 7914

print("Validation completed successfully!")


In [ ]:

# Step 24.4: Preview user-level features

sql_preview = f"""
SELECT
    cohort_week,
    device_type,
    event_count,
    page_views,
    item_views,
    cart_events,
    purchase_events,
    distinct_event_types,
    returned_w1

FROM `{table_name}`

LIMIT 10
"""

df_preview = client.query(
    sql_preview,
    location="US"
).to_dataframe(
    create_bqstorage_client=False
)

print("=== First 10 Feature Records (user IDs excluded) ===")
display(df_preview)


In [ ]:

# Step 25.1: Load the Complete Feature Dataset

import pandas as pd
import numpy as np

# Reuse the configured BigQuery table instead of hard-coding it twice
TABLE_NAME = table_name

sql = f"""
SELECT
    user_id,
    cohort_week,
    cohort_month,
    device_type,
    event_count,
    page_views,
    item_views,
    cart_events,
    checkout_events,
    purchase_events,
    engagement_events,
    distinct_event_types,
    has_session_id,
    has_matched_events,
    returned_w1
FROM `{TABLE_NAME}`
"""

# Query the complete feature table
df = client.query(
    sql,
    location="US"
).to_dataframe(
    create_bqstorage_client=False
)

# Convert dates
df["cohort_week"] = pd.to_datetime(
    df["cohort_week"]
)

print("=== Dataset Overview ===")
print("Shape:", df.shape)
print("Memory usage (MB):",
      round(df.memory_usage(deep=True).sum() / 1024**2, 2))

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== First 5 Rows (user IDs excluded) ===")
display(df.drop(columns=["user_id"]).head())


In [ ]:

# Step 25.2: Data Quality Assessment

print("=== Data Quality Report ===")

# 1. Missing values
missing = df.isna().sum()

print("\nMissing Values:")
print(missing)

# 2. Duplicate users
duplicate_users = df["user_id"].duplicated().sum()

print("\nDuplicate User IDs:", duplicate_users)

# 3. Check target distribution
target_counts = df["returned_w1"].value_counts()

print("\n=== Target Distribution ===")
print(target_counts)

retention_rate = df["returned_w1"].mean() * 100

print(f"\nOverall W1 Retention Rate: {retention_rate:.2f}%")

# 4. Validate event feature consistency
event_cols = [
    "page_views",
    "item_views",
    "cart_events",
    "checkout_events",
    "purchase_events"
]

invalid_subtotals = (
    df[event_cols].sum(axis=1) > df["event_count"]
).sum()

print("\nInvalid Event Subtotals:", invalid_subtotals)

print(
    "Users Without Matched Events:",
    (~df["has_matched_events"].astype(bool)).sum()
)

# 5. Core assertions
assert len(df) == 181360
assert df["user_id"].nunique() == 181360
assert df["returned_w1"].sum() == 7914
assert duplicate_users == 0
assert invalid_subtotals == 0

print("\nCore data quality checks passed!")


In [ ]:

# Step 25.3: Behavioral Feature Distribution

feature_cols = [
    "event_count",
    "page_views",
    "item_views",
    "cart_events",
    "checkout_events",
    "purchase_events",
    "engagement_events",
    "distinct_event_types"
]

print("=== Feature Descriptive Statistics ===")

display(
    df[feature_cols].describe().T
)

print("\n=== Event Count Distribution ===")

print(
    df["event_count"]
    .value_counts()
    .head(15)
)

# Check how many users have exactly 2 events
pct_two_events = (
    df["event_count"].eq(2).mean() * 100
)

print(
    f"\nUsers with exactly 2 events: "
    f"{pct_two_events:.2f}%"
)

# Check features with no variability
unique_counts = df[feature_cols].nunique()

print("\n=== Number of Unique Values ===")
print(unique_counts)

# Check zero-value proportion for each feature
zero_rates = (
    df[feature_cols].eq(0).mean() * 100
)

print("\n=== Percentage of Zero Values ===")
print(zero_rates.round(2))


In [ ]:

# Step 25.4: Cohort-Level Target Distribution

cohort_summary = df.groupby(
    "cohort_week"
).agg(
    total_users=("user_id", "count"),
    retained_users=("returned_w1", "sum"),
    avg_events=("event_count", "mean"),
    avg_page_views=("page_views", "mean")
).reset_index()

cohort_summary["retention_rate_pct"] = (
    cohort_summary["retained_users"] /
    cohort_summary["total_users"] * 100
)

print("=== Cohort-Level Summary ===")

display(cohort_summary.round(2))


In [ ]:

# Step 26: Purged Temporal Train / Validation / Test Split
# Target: Week 1 user retention
# Avoid target-label availability leakage

import pandas as pd
import numpy as np

# 1. Prepare data
df["cohort_week"] = pd.to_datetime(
    df["cohort_week"]
)

df["returned_w1"] = (
    pd.to_numeric(df["returned_w1"])
    .astype(int)
)

# 2. Define time-based cohorts
train_weeks = pd.to_datetime([
    "2020-11-02",
    "2020-11-09",
    "2020-11-16",
    "2020-11-23"
])

val_weeks = pd.to_datetime([
    "2020-12-07"
])

test_weeks = pd.to_datetime([
    "2020-12-21",
    "2020-12-28"
])

# 3. Create time-based splits
train_df = df[
    df["cohort_week"].isin(train_weeks)
].copy()

val_df = df[
    df["cohort_week"].isin(val_weeks)
].copy()

test_df = df[
    df["cohort_week"].isin(test_weeks)
].copy()

# 4. Define model features
numeric_features = [
    "event_count",
    "page_views",
    "item_views",
    "cart_events",
    "checkout_events",
    "purchase_events",
    "engagement_events",
    "distinct_event_types"
]

categorical_features = [
    "device_type"
]

feature_cols = (
    numeric_features + categorical_features
)

target_col = "returned_w1"

# 5. Separate features and targets
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

# 6. Validate no user overlap
train_users = set(train_df["user_id"])
val_users = set(val_df["user_id"])
test_users = set(test_df["user_id"])

assert train_users.isdisjoint(val_users)
assert train_users.isdisjoint(test_users)
assert val_users.isdisjoint(test_users)

# 7. Validate chronological separation
assert (
    train_df["cohort_week"].max()
    < val_df["cohort_week"].min()
)

assert (
    val_df["cohort_week"].max()
    < test_df["cohort_week"].min()
)

# 8. Validate W1 label observation gaps
assert (
    train_df["cohort_week"].max()
    + pd.Timedelta(days=14)
    <= val_df["cohort_week"].min()
)

assert (
    val_df["cohort_week"].max()
    + pd.Timedelta(days=14)
    <= test_df["cohort_week"].min()
)

# 9. Create split summary
split_results = []

for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    split_results.append({
        "split": name,
        "users": len(data),
        "retained_users": int(
            data[target_col].sum()
        ),
        "retention_rate_pct": round(
            data[target_col].mean() * 100, 2
        ),
        "avg_events": round(
            data["event_count"].mean(), 2
        ),
        "avg_page_views": round(
            data["page_views"].mean(), 2
        )
    })

split_summary = pd.DataFrame(split_results)

print("=== Temporal Split Summary ===")
display(split_summary)

print("\n=== Feature Configuration ===")
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

print("\n=== Dataset Shapes ===")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\n=== Data Leakage Checks ===")
print("No overlapping users: PASS")
print("Chronological order: PASS")
print("W1 label observation gaps: PASS")

print("\nTemporal splitting completed successfully.")


In [ ]:

# Step 27: Logistic Regression Baseline
# Predict Week 1 user retention
# Train on historical cohorts and evaluate on validation data

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    FunctionTransformer
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score
)

# 1. Numerical preprocessing
# Impute missing values, apply log1p,
# and standardize numerical features

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    (
        "log_transform",
        FunctionTransformer(
            np.log1p,
            feature_names_out="one-to-one"
        )
    ),
    ("scaler", StandardScaler())
])

# 2. Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(handle_unknown="ignore")
    )
])

# 3. Combine preprocessing pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# 4. Logistic Regression model
logistic_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            penalty="l2",
            solver="lbfgs",
            max_iter=2000,
            random_state=42
        )
    )
])

# 5. Train on training data only
logistic_model.fit(X_train, y_train)

print("Logistic Regression training completed!")

# 6. Predict validation probabilities
val_proba_lr = logistic_model.predict_proba(
    X_val
)[:, 1]

# 7. Main evaluation metrics
val_ap = average_precision_score(
    y_val, val_proba_lr
)

val_roc_auc = roc_auc_score(
    y_val, val_proba_lr
)

baseline_rate = y_val.mean()

# 8. Precision at Top 5%
# Rank users by predicted retention probability
top_fraction = 0.05

k = int(np.ceil(len(y_val) * top_fraction))

top_indices = np.argsort(
    -val_proba_lr
)[:k]

y_val_array = np.asarray(y_val)

precision_at_5 = y_val_array[top_indices].mean()

recall_at_5 = (
    y_val_array[top_indices].sum()
    / y_val_array.sum()
)

lift_at_5 = precision_at_5 / baseline_rate

# 9. Display evaluation results
print("\n=== Logistic Regression Validation Results ===")

print(f"Validation users: {len(y_val):,}")
print(f"Actual retention rate: {baseline_rate:.4%}")

print(f"\nAverage Precision (AP): {val_ap:.4f}")
print(f"ROC-AUC: {val_roc_auc:.4f}")

print(f"\nTop 5% users: {k:,}")
print(f"Precision@5%: {precision_at_5:.4%}")
print(f"Recall@5%: {recall_at_5:.4%}")
print(f"Lift@5%: {lift_at_5:.2f}x")

print(
    f"\nMean predicted retention probability: "
    f"{val_proba_lr.mean():.4%}"
)

# 10. Save results for future model comparison
lr_results = {
    "Model": "Logistic Regression",
    "Average_Precision": val_ap,
    "ROC_AUC": val_roc_auc,
    "Precision_at_5": precision_at_5,
    "Recall_at_5": recall_at_5,
    "Lift_at_5": lift_at_5
}

print("\nBaseline model evaluation completed!")


In [ ]:

# Step 27.2: Precision-Recall Curve
# Logistic Regression validation performance

import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)

# 1. Calculate precision-recall curve
precision, recall, thresholds = precision_recall_curve(
    y_val,
    val_proba_lr
)

ap_score = average_precision_score(
    y_val,
    val_proba_lr
)

# 2. Create visualization
fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    recall,
    precision,
    linewidth=2,
    label=f"Logistic Regression (AP = {ap_score:.4f})"
)

# Random-ranking baseline
ax.axhline(
    y=y_val.mean(),
    linestyle="--",
    linewidth=1.5,
    label=f"Baseline = {y_val.mean():.4f}"
)

# 3. Chart formatting
ax.set_title(
    "Logistic Regression Precision-Recall Curve\n"
    "GA4 Week 1 Retention - Validation Set",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.legend(loc="upper right")
ax.grid(alpha=0.2)

plt.tight_layout()

# 4. Save figure
plt.savefig(
    "ga4_logistic_regression_pr_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Precision-Recall curve saved successfully.")


In [ ]:

# Step 28.1: Install LightGBM

%pip -q install lightgbm


In [ ]:

# Step 28.2: Train LightGBM
# Use the same temporal split as Logistic Regression
# Do not use the test set

import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss
)

# 1. Preprocessing for tree-based models
# Scaling and log transformation are not required
# for LightGBM in this baseline comparison.

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_lgb = ColumnTransformer([
    ("num", num_pipeline, numeric_features),
    ("cat", cat_pipeline, categorical_features)
])

# 2. Fit preprocessing on training data only
X_train_lgb = preprocessor_lgb.fit_transform(X_train)
X_val_lgb = preprocessor_lgb.transform(X_val)

# 3. Initialize LightGBM
lgb_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=15,
    max_depth=-1,
    min_child_samples=120,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.9,
    reg_lambda=2.0,
    metric="average_precision",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# 4. Train with validation-based early stopping
lgb_model.fit(
    X_train_lgb,
    y_train,
    eval_set=[(X_val_lgb, y_val)],
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=False
        )
    ]
)

print("LightGBM training completed!")
print("Best iteration:", lgb_model.best_iteration_)

# 5. Predict validation probabilities
val_proba_lgb = lgb_model.predict_proba(
    X_val_lgb
)[:, 1]

# 6. Calculate validation metrics
y_true = np.asarray(y_val, dtype=int)

def evaluate_model(name, y_true, probabilities):
    ap = average_precision_score(y_true, probabilities)
    roc = roc_auc_score(y_true, probabilities)
    brier = brier_score_loss(y_true, probabilities)
    logloss = log_loss(y_true, probabilities)

    k = int(np.ceil(len(y_true) * 0.05))
    top_indices = np.argsort(-probabilities)[:k]

    precision_at_5 = y_true[top_indices].mean()
    recall_at_5 = (
        y_true[top_indices].sum() / y_true.sum()
    )
    lift_at_5 = (
        precision_at_5 / y_true.mean()
    )

    return {
        "Model": name,
        "AP": ap,
        "ROC_AUC": roc,
        "Precision_at_5": precision_at_5,
        "Recall_at_5": recall_at_5,
        "Lift_at_5": lift_at_5,
        "Brier_Score": brier,
        "Log_Loss": logloss,
        "Mean_Predicted_Rate": probabilities.mean()
    }

# 7. Compare both models on the same validation set
lr_metrics = evaluate_model(
    "Logistic Regression",
    y_true,
    val_proba_lr
)

lgb_metrics = evaluate_model(
    "LightGBM",
    y_true,
    val_proba_lgb
)

comparison_df = pd.DataFrame([
    lr_metrics,
    lgb_metrics
]).set_index("Model")

print("\n=== Validation Model Comparison ===")
display(comparison_df.round(4))

print("\n=== Retention Baseline ===")
print(f"Actual retention rate: {y_true.mean():.4%}")

print("\n=== Prediction Summary ===")
print(f"LR mean probability: {val_proba_lr.mean():.4%}")
print(f"LGBM mean probability: {val_proba_lgb.mean():.4%}")

# Save comparison table
comparison_df.to_csv(
    "ga4_model_validation_comparison.csv"
)

print("\nModel comparison saved successfully.")


In [ ]:

# Step 28.3: Compare Precision-Recall Curves

import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)

fig, ax = plt.subplots(figsize=(10, 6))

models = {
    "Logistic Regression": val_proba_lr,
    "LightGBM": val_proba_lgb
}

for name, probabilities in models.items():

    precision, recall, _ = precision_recall_curve(
        y_val, probabilities
    )

    ap = average_precision_score(
        y_val, probabilities
    )

    ax.plot(
        recall,
        precision,
        linewidth=2,
        label=f"{name} (AP = {ap:.4f})"
    )

# Random ranking baseline
ax.axhline(
    y=y_val.mean(),
    linestyle="--",
    linewidth=1.5,
    label=f"Baseline = {y_val.mean():.4f}"
)

ax.set_title(
    "Logistic Regression vs LightGBM\n"
    "GA4 W1 Retention - Validation Set",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

plt.savefig(
    "ga4_lr_vs_lightgbm_pr_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# Step 29.1: Install SHAP

%pip -q install shap


In [ ]:

# Step 29.2: Explain LightGBM Using SHAP
# Use validation data only for model interpretation

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Get transformed feature names
feature_names = (
    preprocessor_lgb.get_feature_names_out()
)

feature_names = [
    name.replace("num__", "").replace("cat__", "")
    for name in feature_names
]

# 2. Sample validation records
SAMPLE_SIZE = 1000
rng = np.random.default_rng(42)

sample_idx = rng.choice(
    X_val_lgb.shape[0],
    size=min(SAMPLE_SIZE, X_val_lgb.shape[0]),
    replace=False
)

X_shap = X_val_lgb[sample_idx]

# Convert sparse matrix if necessary
if hasattr(X_shap, "toarray"):
    X_shap = X_shap.toarray()

# Create DataFrame with readable feature names
X_shap_df = pd.DataFrame(
    X_shap,
    columns=feature_names
)

# 3. Create SHAP TreeExplainer
explainer = shap.TreeExplainer(
    lgb_model,
    model_output="raw"
)

# 4. Calculate SHAP values
raw_values = explainer.shap_values(X_shap)

# Handle different SHAP output formats
if isinstance(raw_values, list):
    shap_values = raw_values[-1]
else:
    shap_values = np.asarray(raw_values)

if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]

assert shap_values.shape == X_shap_df.shape

print("SHAP calculation completed!")
print("SHAP shape:", shap_values.shape)

# 5. Calculate global feature importance
shap_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(
        shap_values
    ).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    "mean_abs_shap",
    ascending=False
)

print("\n=== SHAP Feature Importance ===")
display(shap_importance)

# 6. Save results
shap_importance.to_csv(
    "ga4_shap_feature_importance.csv",
    index=False
)


In [ ]:

# Step 29.3: SHAP Summary Plot

import matplotlib.pyplot as plt

plt.figure()

shap.summary_plot(
    shap_values,
    X_shap_df,
    max_display=12,
    show=False
)

plt.title(
    "LightGBM SHAP Summary\n"
    "GA4 W1 Retention Prediction",
    fontsize=13,
    pad=18
)

plt.tight_layout()

plt.savefig(
    "ga4_lightgbm_shap_summary.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("SHAP summary plot saved successfully.")


In [ ]:

# Step 29.4: SHAP Global Feature Importance

plt.figure(figsize=(10, 6))

shap.summary_plot(
    shap_values,
    X_shap_df,
    plot_type="bar",
    max_display=12,
    show=False
)

plt.title(
    "LightGBM Global Feature Importance\n"
    "Mean Absolute SHAP Values",
    fontsize=13,
    pad=18
)

plt.tight_layout()

plt.savefig(
    "ga4_lightgbm_shap_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("SHAP importance chart saved successfully.")


In [ ]:

# Step 30: Probability Calibration Diagnostics
# Compare Logistic Regression and LightGBM
# Validation set only - do not use the test set

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    average_precision_score
)

# 1. Prepare validation labels and probabilities
y_true = np.asarray(y_val, dtype=int)

model_probabilities = {
    "Logistic Regression": np.asarray(
        val_proba_lr, dtype=float
    ),
    "LightGBM": np.asarray(
        val_proba_lgb, dtype=float
    )
}

# 2. Create equal-frequency calibration bins
def calibration_diagnostics(y_true, probabilities, n_bins=10):

    data = pd.DataFrame({
        "actual": y_true,
        "predicted": probabilities
    })

    # Sort from low to high predicted probability
    data = data.sort_values(
        "predicted"
    ).reset_index(drop=True)

    # Assign approximately equal-sized bins
    data["bin"] = (
        np.arange(len(data)) * n_bins
        // len(data)
    ) + 1

    summary = data.groupby("bin").agg(
        users=("actual", "size"),
        actual_rate=("actual", "mean"),
        predicted_rate=("predicted", "mean")
    ).reset_index()

    # Expected Calibration Error
    summary["absolute_error"] = (
        summary["actual_rate"]
        - summary["predicted_rate"]
    ).abs()

    ece = (
        summary["absolute_error"]
        * summary["users"]
    ).sum() / len(data)

    return summary, ece


# 3. Evaluate model calibration
calibration_results = {}
metric_rows = []

for name, probabilities in model_probabilities.items():

    assert len(probabilities) == len(y_true)
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) &
            (probabilities <= 1)).all()

    summary, ece = calibration_diagnostics(
        y_true,
        probabilities,
        n_bins=10
    )

    calibration_results[name] = summary

    metric_rows.append({
        "Model": name,
        "Actual_Rate": y_true.mean(),
        "Mean_Predicted": probabilities.mean(),
        "AP": average_precision_score(
            y_true, probabilities
        ),
        "Brier_Score": brier_score_loss(
            y_true, probabilities
        ),
        "Log_Loss": log_loss(
            y_true, probabilities
        ),
        "ECE": ece
    })

metrics_df = pd.DataFrame(metric_rows)

print("=== Probability Calibration Metrics ===")
display(metrics_df.round(5))

# 4. Reliability diagram
fig, ax = plt.subplots(figsize=(9, 7))

for name, summary in calibration_results.items():

    ax.plot(
        summary["predicted_rate"] * 100,
        summary["actual_rate"] * 100,
        marker="o",
        linewidth=2,
        label=name
    )

max_rate = max(
    max(s["predicted_rate"].max(),
        s["actual_rate"].max())
    for s in calibration_results.values()
) * 100

upper = max_rate * 1.15

ax.plot(
    [0, upper],
    [0, upper],
    linestyle="--",
    color="gray",
    label="Perfect Calibration"
)

ax.set_xlim(0, upper)
ax.set_ylim(0, upper)

ax.set_xlabel("Mean Predicted Probability (%)")
ax.set_ylabel("Observed W1 Retention Rate (%)")

ax.set_title(
    "Probability Calibration Comparison\n"
    "GA4 W1 Retention - Validation Set",
    fontsize=14,
    fontweight="bold"
)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

plt.savefig(
    "ga4_probability_calibration.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# 5. Detailed LightGBM calibration table
print("\n=== LightGBM Calibration by Decile ===")

lgb_calibration = calibration_results["LightGBM"].copy()

lgb_calibration["predicted_rate_pct"] = (
    lgb_calibration["predicted_rate"] * 100
)

lgb_calibration["actual_rate_pct"] = (
    lgb_calibration["actual_rate"] * 100
)

display(
    lgb_calibration[[
        "bin",
        "users",
        "predicted_rate_pct",
        "actual_rate_pct"
    ]].round(3)
)

# 6. Save results
metrics_df.to_csv(
    "ga4_calibration_metrics.csv",
    index=False
)

lgb_calibration.to_csv(
    "ga4_lightgbm_calibration_deciles.csv",
    index=False
)

print("\nCalibration diagnostics completed!")


In [ ]:

# Step 31: Final LightGBM Test Evaluation
# Evaluate the frozen model on the untouched test set
# No retraining, tuning, or calibration

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    precision_recall_curve
)

# 1. Verify test data
y_test_array = np.asarray(y_test, dtype=int)

assert len(X_test) == 33752
assert y_test_array.sum() == 840

print("=== Final Test Dataset ===")
print("Test users:", len(X_test))
print("Retained users:", y_test_array.sum())
print("Actual retention rate:", f"{y_test_array.mean():.4%}")

# 2. Apply training-fitted preprocessing
X_test_lgb = preprocessor_lgb.transform(X_test)

# 3. Predict using the frozen LightGBM model
test_proba_lgb = lgb_model.predict_proba(
    X_test_lgb,
    num_iteration=lgb_model.best_iteration_
)[:, 1]

assert np.isfinite(test_proba_lgb).all()
assert np.all(
    (test_proba_lgb >= 0) &
    (test_proba_lgb <= 1)
)

# 4. Ranking performance
test_ap = average_precision_score(
    y_test_array, test_proba_lgb
)

test_roc = roc_auc_score(
    y_test_array, test_proba_lgb
)

# 5. Precision and Recall at Top 5%
k = int(np.ceil(len(y_test_array) * 0.05))

top_indices = np.argsort(
    -test_proba_lgb,
    kind="stable"
)[:k]

test_precision_5 = y_test_array[top_indices].mean()

test_recall_5 = (
    y_test_array[top_indices].sum() /
    y_test_array.sum()
)

test_lift_5 = (
    test_precision_5 / y_test_array.mean()
)

# 6. Probability prediction performance
test_brier = brier_score_loss(
    y_test_array, test_proba_lgb
)

test_logloss = log_loss(
    y_test_array, test_proba_lgb
)

# 7. Expected Calibration Error
def calculate_ece(y_true, probabilities, n_bins=10):

    order = np.argsort(probabilities, kind="stable")
    groups = np.array_split(order, n_bins)

    weighted_error = 0.0

    for idx in groups:
        actual = y_true[idx].mean()
        predicted = probabilities[idx].mean()

        weighted_error += (
            len(idx) * abs(actual - predicted)
        )

    return weighted_error / len(y_true)

test_ece = calculate_ece(
    y_test_array, test_proba_lgb
)

# 8. Final test metrics
test_results = pd.DataFrame([{
    "Model": "LightGBM",
    "Dataset": "Test",
    "AP": test_ap,
    "ROC_AUC": test_roc,
    "Precision_at_5": test_precision_5,
    "Recall_at_5": test_recall_5,
    "Lift_at_5": test_lift_5,
    "Brier_Score": test_brier,
    "Log_Loss": test_logloss,
    "ECE": test_ece,
    "Actual_Retention": y_test_array.mean(),
    "Mean_Predicted": test_proba_lgb.mean()
}])

print("\n=== Final LightGBM Test Results ===")
display(test_results.round(5))

print("\n=== Key Business Metrics ===")
print(f"Test Average Precision: {test_ap:.4f}")
print(f"ROC-AUC: {test_roc:.4f}")
print(f"Precision@5%: {test_precision_5:.4%}")
print(f"Recall@5%: {test_recall_5:.4%}")
print(f"Lift@5%: {test_lift_5:.2f}x")
print(f"Test baseline: {y_test_array.mean():.4%}")
print(f"Mean predicted probability: {test_proba_lgb.mean():.4%}")

# 9. Save evaluation results
test_results.to_csv(
    "ga4_lightgbm_final_test_metrics.csv",
    index=False
)

print("\nFinal test evaluation completed!")


In [ ]:

# Step 31.2: Final Test Precision-Recall Curve

precision, recall, _ = precision_recall_curve(
    y_test_array,
    test_proba_lgb
)

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    recall,
    precision,
    linewidth=2,
    label=f"LightGBM Test AP = {test_ap:.4f}"
)

ax.axhline(
    y=y_test_array.mean(),
    linestyle="--",
    linewidth=1.5,
    label=f"Random baseline = {y_test_array.mean():.4f}"
)

ax.set_title(
    "LightGBM Final Test Precision-Recall Curve\n"
    "GA4 Week 1 Retention Prediction",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

plt.savefig(
    "ga4_lightgbm_final_test_pr_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# Step 32.1: Temporal Prediction Drift Analysis
# Analyze validation and test cohorts
# No model retraining or parameter tuning

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import brier_score_loss

# Prepare cohort-level predictions
val_diagnostic = val_df[
    ["cohort_week", "returned_w1"]
].reset_index(drop=True).copy()

val_diagnostic["predicted"] = val_proba_lgb
val_diagnostic["dataset"] = "Validation"

test_diagnostic = test_df[
    ["cohort_week", "returned_w1"]
].reset_index(drop=True).copy()

test_diagnostic["predicted"] = test_proba_lgb
test_diagnostic["dataset"] = "Test"

diagnostic_df = pd.concat(
    [val_diagnostic, test_diagnostic],
    ignore_index=True
)

# Calculate cohort-level calibration
cohort_drift = diagnostic_df.groupby(
    ["dataset", "cohort_week"],
    as_index=False
).agg(
    users=("returned_w1", "size"),
    retained_users=("returned_w1", "sum"),
    actual_rate=("returned_w1", "mean"),
    predicted_rate=("predicted", "mean")
)

# Display cohorts in actual calendar order (Validation before Test)
cohort_drift = cohort_drift.sort_values(
    "cohort_week"
).reset_index(drop=True)

cohort_drift["actual_rate_pct"] = (
    cohort_drift["actual_rate"] * 100
)

cohort_drift["predicted_rate_pct"] = (
    cohort_drift["predicted_rate"] * 100
)

cohort_drift["overprediction_pp"] = (
    cohort_drift["predicted_rate_pct"]
    - cohort_drift["actual_rate_pct"]
)

print("=== Temporal Calibration Drift ===")

display(
    cohort_drift[[
        "dataset",
        "cohort_week",
        "users",
        "retained_users",
        "actual_rate_pct",
        "predicted_rate_pct",
        "overprediction_pp"
    ]].round(3)
)

# Plot actual vs predicted retention by cohort
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(cohort_drift))
width = 0.35

ax.bar(
    x - width / 2,
    cohort_drift["actual_rate_pct"],
    width,
    label="Actual Retention"
)

ax.bar(
    x + width / 2,
    cohort_drift["predicted_rate_pct"],
    width,
    label="Mean Predicted Retention"
)

labels = [
    f"{row['cohort_week']:%Y-%m-%d}\n{row['dataset']}"
    for _, row in cohort_drift.iterrows()
]

ax.set_xticks(x)
ax.set_xticklabels(labels)

ax.set_ylabel("W1 Retention Rate (%)")

ax.set_title(
    "LightGBM Temporal Calibration Drift\n"
    "Actual vs Predicted W1 Retention"
)

ax.legend()
plt.tight_layout()

plt.savefig(
    "ga4_temporal_calibration_drift.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Compare model with historical-rate baseline
historical_rate = y_train.mean()

print("\n=== Historical Baseline Comparison ===")

for name, y, probabilities in [
    ("Validation", y_val, val_proba_lgb),
    ("Test", y_test, test_proba_lgb)
]:

    y_array = np.asarray(y, dtype=int)

    historical_predictions = np.full(
        len(y_array),
        historical_rate
    )

    print(f"\n{name}")
    print(
        "LightGBM Brier:",
        round(brier_score_loss(y_array, probabilities), 5)
    )
    print(
        "Historical baseline Brier:",
        round(brier_score_loss(
            y_array, historical_predictions
        ), 5)
    )

cohort_drift.to_csv(
    "ga4_temporal_drift_summary.csv",
    index=False
)

print("\nTemporal calibration analysis completed.")


In [ ]:

# Step 32.2: Behavioral Feature Drift
# Compare feature distributions across time

import pandas as pd
import numpy as np

records = []

for dataset_name, dataset in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:

    for feature in numeric_features:

        values = pd.to_numeric(
            dataset[feature],
            errors="coerce"
        )

        records.append({
            "dataset": dataset_name,
            "feature": feature,
            "mean": values.mean(),
            "median": values.median(),
            "zero_pct": values.eq(0).mean() * 100
        })

feature_drift = pd.DataFrame(records)

mean_comparison = feature_drift.pivot(
    index="feature",
    columns="dataset",
    values="mean"
)[["Train", "Validation", "Test"]]

mean_comparison["test_vs_train_change_pct"] = (
    (
        mean_comparison["Test"]
        / mean_comparison["Train"] - 1
    ) * 100
)

print("=== Feature Distribution Drift ===")
display(mean_comparison.round(3))

print("\n=== Feature Zero Rate by Dataset ===")

zero_comparison = feature_drift.pivot(
    index="feature",
    columns="dataset",
    values="zero_pct"
)[["Train", "Validation", "Test"]]

display(zero_comparison.round(2))

# Save drift diagnostics
mean_comparison.to_csv(
    "ga4_feature_drift.csv"
)

print("\nFeature drift analysis completed.")


In [ ]:

# Step 33: GA4 Interactive Model Monitoring Dashboard
# Create a standalone interactive HTML dashboard

%pip -q install plotly

import pandas as pd
import numpy as np
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from sklearn.metrics import average_precision_score

# 1. Weekly cohort retention trend
cohort = df.groupby("cohort_week").agg(
    users=("returned_w1", "size"),
    retained=("returned_w1", "sum")
).reset_index()

cohort["retention_pct"] = (
    cohort["retained"] / cohort["users"] * 100
)

cohort = cohort.sort_values("cohort_week")

# 2. Behavior feature drift
selected_features = [
    "page_views",
    "event_count",
    "engagement_events",
    "item_views",
    "distinct_event_types"
]

train_means = train_df[selected_features].mean()
test_means = test_df[selected_features].mean()

feature_change = (
    (test_means / train_means - 1) * 100
).sort_values()

# 3. Cohort-level prediction calibration
validation_predictions = pd.DataFrame({
    "cohort_week": val_df["cohort_week"].to_numpy(),
    "actual": np.asarray(y_val),
    "predicted": val_proba_lgb
})

validation_predictions["dataset"] = "Validation"

test_predictions = pd.DataFrame({
    "cohort_week": test_df["cohort_week"].to_numpy(),
    "actual": np.asarray(y_test),
    "predicted": test_proba_lgb
})

test_predictions["dataset"] = "Test"

prediction_data = pd.concat([
    validation_predictions,
    test_predictions
], ignore_index=True)

prediction_summary = prediction_data.groupby(
    ["cohort_week", "dataset"],
    as_index=False
).agg(
    actual_rate=("actual", "mean"),
    predicted_rate=("predicted", "mean")
)

prediction_summary = prediction_summary.sort_values(
    "cohort_week"
)

prediction_summary["actual_rate"] *= 100
prediction_summary["predicted_rate"] *= 100

# 4. Validation and test AP
performance = pd.DataFrame({
    "dataset": ["Validation", "Test"],
    "AP": [
        average_precision_score(y_val, val_proba_lgb),
        average_precision_score(y_test, test_proba_lgb)
    ],
    "baseline": [
        np.asarray(y_val).mean(),
        np.asarray(y_test).mean()
    ]
})

# 5. Create dashboard layout
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Weekly Cohort Retention",
        "Behavior Feature Drift",
        "Actual vs Predicted Retention",
        "Model Average Precision"
    ],
    vertical_spacing=0.17,
    horizontal_spacing=0.13
)

# Chart 1: Weekly retention
fig.add_trace(
    go.Scatter(
        x=cohort["cohort_week"],
        y=cohort["retention_pct"],
        mode="lines+markers",
        name="W1 Retention",
        hovertemplate="%{x}<br>%{y:.2f}%<extra></extra>"
    ),
    row=1, col=1
)

# Chart 2: Feature drift
fig.add_trace(
    go.Bar(
        x=feature_change.values,
        y=feature_change.index,
        orientation="h",
        name="Feature Change (%)",
        hovertemplate="%{y}<br>%{x:.2f}%<extra></extra>"
    ),
    row=1, col=2
)

# Chart 3: Calibration drift
date_labels = (
    prediction_summary["cohort_week"]
    .dt.strftime("%Y-%m-%d")
)

fig.add_trace(
    go.Bar(
        x=date_labels,
        y=prediction_summary["actual_rate"],
        name="Actual Retention"
    ),
    row=2, col=1
)

fig.add_trace(
    go.Bar(
        x=date_labels,
        y=prediction_summary["predicted_rate"],
        name="Predicted Retention"
    ),
    row=2, col=1
)

# Chart 4: AP comparison
fig.add_trace(
    go.Bar(
        x=performance["dataset"],
        y=performance["AP"],
        name="LightGBM AP"
    ),
    row=2, col=2
)

fig.add_trace(
    go.Bar(
        x=performance["dataset"],
        y=performance["baseline"],
        name="Random Baseline"
    ),
    row=2, col=2
)

# 6. Customize dashboard
fig.update_layout(
    title={
        "text": (
            "GA4 E-commerce Analytics"
            "<br><sup>Retention and Model Monitoring Dashboard</sup>"
        ),
        "x": 0.5,
        "xanchor": "center"
    },
    template="plotly_white",
    height=900,
    width=1250,
    barmode="group",
    legend=dict(
        orientation="h",
        y=-0.1,
        x=0.5,
        xanchor="center"
    ),
    margin=dict(
        l=90, r=60, t=140, b=100
    )
)

fig.update_yaxes(
    title_text="Retention Rate (%)",
    row=1, col=1
)

fig.update_xaxes(
    title_text="Test vs Train Change (%)",
    row=1, col=2
)

fig.update_yaxes(
    title_text="Retention Rate (%)",
    row=2, col=1
)

fig.update_yaxes(
    title_text="Average Precision",
    row=2, col=2
)

# 7. Display interactive dashboard
fig.show()

# 8. Export standalone HTML
output_file = "GA4_Model_Monitoring_Dashboard.html"

fig.write_html(
    output_file,
    include_plotlyjs=True,
    full_html=True
)

print("Dashboard created successfully!")
print("Saved file:", output_file)


In [ ]:

# Download the interactive dashboard

from google.colab import files

files.download("GA4_Model_Monitoring_Dashboard.html")
